In [0]:
"""
Databricks SQL File Executor
A utility module for reading SQL files and executing them with parameter substitution.
"""

from typing import Dict, List, Optional, Any
import os
import re
from pathlib import Path

In [0]:
class DatabricksSQLExecutor:
    """
    Execute SQL statements from files with parameter substitution.
    Supports named parameters in SQL using ${parameter_name} syntax.
    """

    def __init__(self, spark):
        """
        Initialize the executor with a Spark session.
        
        Args:
            spark: The Spark session (automatically available in Databricks notebooks)
        """
        self.spark = spark

    def read_sql_file(self, file_path) -> str:
        """
        Read a SQL file from the filesystem.
        
        Args:
            file_path (str or Path): Path to the SQL file (supports DBFS paths like /dbfs/path/to/file.sql)
        
        Returns:
            str: The SQL content
            
        Raises:
            FileNotFoundError: If the file doesn't exist
        """
        # Convert Path object to string if needed
        file_path_str = str(file_path)
        
        # Handle DBFS paths
        if file_path_str.startswith("/dbfs/"):
            local_path = file_path_str
        elif file_path_str.startswith("dbfs:"):
            local_path = file_path_str.replace("dbfs:", "/dbfs")
        else:
            local_path = file_path_str

        try:
            with open(local_path, 'r') as f:
                return f.read()
        except FileNotFoundError:
            raise FileNotFoundError(f"SQL file not found: {file_path_str}")

    def substitute_parameters(self, sql: str, parameters: Dict[str, Any]) -> str:
        """
        Replace parameter placeholders in SQL with actual values.
        Supports ${parameter_name} syntax.
        
        Args:
            sql (str): The SQL statement with placeholders
            parameters (Dict[str, Any]): Dictionary of parameter names and values
        
        Returns:
            str: The SQL statement with substituted values
            
        Example:
            sql = "SELECT * FROM users WHERE id = ${user_id} AND name = '${user_name}'"
            params = {"user_id": 123, "user_name": "John"}
            result = executor.substitute_parameters(sql, params)
        
        Note:
            - Parameters in identifier positions (table names, schema names) are NOT quoted
            - Parameters in value positions should already be quoted in the SQL template
            - Example: SELECT * FROM ${schema}.${table} WHERE id = '${id}'
                      NOT: SELECT * FROM '${schema}'.'${table}' WHERE id = '${id}'
        """
        result = sql
        
        for key, value in parameters.items():
            placeholder = f"${{{key}}}"
            
            # Handle different value types
            # NOTE: We don't automatically add quotes - let the SQL template decide
            if isinstance(value, bool):
                substituted_value = "true" if value else "false"
            elif value is None:
                substituted_value = "NULL"
            elif isinstance(value, (int, float)):
                substituted_value = str(value)
            else:
                # For strings and other types, convert to string without adding quotes
                # The SQL template should include quotes if needed
                substituted_value = str(value)
            
            result = result.replace(placeholder, substituted_value)
        
        return result

    def execute_sql(self, sql, parameters: Optional[Dict[str, Any]] = None) -> Any:
        """
        Execute a SQL statement or file with optional parameter substitution.
        
        Args:
            sql (str or Path): The SQL statement to execute, or path to a SQL file
            parameters (Optional[Dict[str, Any]]): Dictionary of parameters to substitute
        
        Returns:
            DataFrame: The result of the SQL execution
        """
        # Convert to string if it's a Path object
        sql_str = str(sql)
        
        # Check if this is a file path (not a SQL statement)
        # If it ends with .sql or looks like a path, treat it as a file
        if sql_str.endswith('.sql') or sql_str.startswith('/Workspace') or sql_str.startswith('/dbfs') or sql_str.startswith('dbfs:'):
            # It's a file path, read the file first
            sql = self.read_sql_file(sql)
        else:
            sql = sql_str
        
        if parameters:
            sql = self.substitute_parameters(sql, parameters)
        
        print(f"Executing SQL:\n{sql}\n")
        return self.spark.sql(sql)

    def execute_sql_file(self, file_path, parameters: Optional[Dict[str, Any]] = None) -> Any:
        """
        Read and execute a SQL file with optional parameter substitution.
        
        Args:
            file_path (str or Path): Path to the SQL file
            parameters (Optional[Dict[str, Any]]): Dictionary of parameters to substitute
        
        Returns:
            DataFrame: The result of the SQL execution
        """
        sql = self.read_sql_file(file_path)
        return self.execute_sql(sql, parameters)

    def execute_sql_file_multi_statement(
        self, 
        file_path, 
        parameters: Optional[Dict[str, Any]] = None
    ) -> List[Any]:
        """
        Execute a SQL file containing multiple statements (separated by semicolons).
        
        Args:
            file_path (str or Path): Path to the SQL file
            parameters (Optional[Dict[str, Any]]): Dictionary of parameters to substitute
        
        Returns:
            List[Any]: List of results from each statement
        """
        sql = self.read_sql_file(file_path)
        
        if parameters:
            sql = self.substitute_parameters(sql, parameters)
        
        # Split by semicolon, filter out empty statements
        statements = [stmt.strip() for stmt in sql.split(';') if stmt.strip()]
        
        results = []
        for i, statement in enumerate(statements, 1):
            print(f"Executing statement {i}:\n{statement}\n")
            try:
                result = self.spark.sql(statement)
                results.append(result)
            except Exception as e:
                print(f"Error executing statement {i}: {str(e)}")
                results.append(None)
        
        return results

    def validate_parameters(self, sql: str, parameters: Dict[str, Any]) -> List[str]:
        """
        Validate that all required parameters are provided.
        
        Args:
            sql (str): The SQL statement
            parameters (Dict[str, Any]): Dictionary of provided parameters
        
        Returns:
            List[str]: List of missing parameters (empty if all are provided)
        """
        # Find all parameter placeholders
        placeholders = re.findall(r'\$\{(\w+)\}', sql)
        required_params = set(placeholders)
        provided_params = set(parameters.keys())
        
        missing = required_params - provided_params
        return sorted(list(missing))